# SigLIP2 finetuning

**Full pipeline (CIFAR test):** run cells **1 → 6 in order** — env → config → checkpoint → sanity → **train** → **eval**.

**Pipeline test (default):** `TRAINING_MODE = "legacy_class"` in [`finetune_config.py`](finetune_config.py) uses your CIFAR-10 pickles — no JSONL needed. Training sees all **50k** train images; eval uses all **10k** test images.

**Real captions later:** set `TRAINING_MODE = "text"` and add `data/train.jsonl`.

Text-only stage: image tower frozen, **text tower** + `t` + `b` train.

## 1. Environment (same as `set_up.ipynb`)

**Before you start:** close `set_up.ipynb` and **Restart Kernel** here. If `nvidia-smi` shows a `python` process using >6 GB, that stale process will block JAX during training (cell 5) or eval (cell 6).

Then run all cells **1 → 6** without skipping cell 5 (training).

In [1]:
# Environment setup — same as set_up.ipynb (SigLIP 2 Colab local cell)
import os, shutil, subprocess, sys
from pathlib import Path

# Save project root before chdir into big_vision/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'finetune_config.py').exists():
    raise RuntimeError('Run this notebook from the Siglip2_test project root.')

# Hide GPU from TensorFlow (JAX uses it). Same as set_up.ipynb.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.95')
os.environ.pop('BV_JAX_INIT', None)  # TPU-only; must not be set on single GPU

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
assert tf.config.get_visible_devices('GPU') == [], 'TF should not see any GPU'
print('TF', tf.__version__, '— GPU hidden, will run on CPU')

if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)

if not os.path.isdir('big_vision'):
    subprocess.run(
        ['git', 'clone', '--quiet', '--branch=main', '--depth=1',
         'https://github.com/google-research/big_vision'],
        check=True,
    )

if os.path.basename(os.getcwd()) != 'big_vision':
    os.chdir('big_vision')
print('cwd:', os.getcwd())

# Do NOT import JAX in this notebook until the eval cell (after training).
# Training runs in a fresh subprocess (cell 5) — same trick as avoiding TF+JAX clash.
print('Ready. JAX will load in cell 5 (trainer subprocess) and cell 6 (eval).')

TF 2.20.0 — GPU hidden, will run on CPU
Wed May 20 20:08:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.07             Driver Version: 570.133.07     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070 Ti     Off |   00000000:B3:00.0  On |                  N/A |
|  0%   41C    P3             17W /  310W |     419MiB /   8192MiB |     16%      Default |
|                                         |                        |                  N/A |
+-------

## 2. Load settings from `finetune_config.py`

In [2]:
import importlib
sys.path.insert(0, str(PROJECT_ROOT))
import finetune_config as cfg
importlib.reload(cfg)

print('Training mode:', cfg.TRAINING_MODE)
print('Model:', cfg.VARIANT, '@', cfg.RES)
print('Steps:', cfg.TOTAL_STEPS, '| batch:', cfg.BATCH_SIZE,
      f'(~{cfg.TOTAL_STEPS * cfg.BATCH_SIZE // 1000}k image-caption pairs)')
print('Fresh train (wipe workdir):', cfg.FRESH_TRAIN)
print('Freeze image tower (text-only):', cfg.FREEZE_IMAGE_TOWER)
print('Checkpoint:', cfg.checkpoint_path())
print('Workdir:', cfg.WORKDIR)
eval_n = cfg.EVAL_MAX_IMAGES
print('Eval images:', 'all 10k test' if eval_n is None else eval_n)

os.environ['SIGLIP_DATASET_MODE'] = cfg.dataset_mode_for_trainer()
os.environ['SIGLIP_CKPT_PATH'] = str(cfg.checkpoint_path())
os.environ['SIGLIP_FREEZE_IMAGE'] = '1' if cfg.FREEZE_IMAGE_TOWER else '0'

if cfg.TRAINING_MODE == 'legacy_class':
    if not cfg.CIFAR_ROOT.is_dir():
        raise FileNotFoundError(f'CIFAR not found: {cfg.CIFAR_ROOT}')
    os.environ['SIGLIP_DATASET_ROOT'] = str(cfg.CIFAR_ROOT)
    os.environ['SIGLIP_CIFAR_MULTI_PROMPT'] = '1'
    print('Pipeline test: CIFAR-10 at', cfg.CIFAR_ROOT)
    print('Caption templates:', cfg.CIFAR_PROMPTS)
else:
    train_jsonl = cfg.TEXT_JSONL_TRAIN
    if not train_jsonl.is_file():
        raise FileNotFoundError(
            f'Missing {train_jsonl}. Copy data/train.jsonl.example → data/train.jsonl '
            'and add your image paths + captions.'
        )
    os.environ['SIGLIP_JSONL_TRAIN'] = str(train_jsonl)
    os.environ['SIGLIP_JSONL_VAL'] = str(cfg.TEXT_JSONL_VAL)
    if cfg.IMAGE_ROOT.is_dir():
        os.environ['SIGLIP_IMAGE_ROOT'] = str(cfg.IMAGE_ROOT)
    print('Caption JSONL:', train_jsonl)

Training mode: legacy_class
Model: B/16 @ 224
Steps: 3000 | batch: 16 (~48k image-caption pairs)
Fresh train (wipe workdir): True
Freeze image tower (text-only): True
Checkpoint: /tmp/siglip2_b16_224.npz
Workdir: /home/valenbonas/Documents/Investigacion_doctorado/Siglip2_test/workdirs/siglip2_cifar_test
Eval images: all 10k test
Pipeline test: CIFAR-10 at /home/valenbonas/Documents/Investigacion_doctorado/Siglip2_test/datasets/cifar-10-batches-py
Caption templates: ('a photo of a {class_name}', 'a picture of a {class_name}', 'an image showing a {class_name}')


## 3. Download SigLIP2 checkpoint (if needed)

In [3]:
ckpt = cfg.checkpoint_path()
ckpt.parent.mkdir(parents=True, exist_ok=True)
if not ckpt.is_file():
    remote = f'gs://big_vision/siglip2/{ckpt.name}'
    print('Downloading', remote, '→', ckpt)
    subprocess.run(['gsutil', 'cp', remote, str(ckpt)], check=True)
else:
    print('Checkpoint already present:', ckpt)

Checkpoint already present: /tmp/siglip2_b16_224.npz


## 4. Sanity-check dataset (one batch of pairs)

In [4]:
# Quick CIFAR check without importing JAX (avoids GPU grab before training)
import pickle

def _cifar_preview(root, n=3):
    meta = pickle.load(open(root / 'batches.meta', 'rb'), encoding='bytes')
    names = [x.decode() for x in meta[b'label_names']]
    batch = pickle.load(open(root / 'data_batch_1', 'rb'), encoding='bytes')
    for i in range(n):
        lab = batch[b'labels'][i]
        print(f'  [{i}] label={lab} ({names[lab]})  caption: "a photo of a {names[lab]}"')
    return len(names), sum(1 for _ in range(1, 6)) * 10000

if cfg.TRAINING_MODE == 'legacy_class':
    nclass, ntrain = _cifar_preview(cfg.CIFAR_ROOT)
    print(f'CIFAR OK: {nclass} classes, {ntrain} train images')
else:
    print('JSONL train:', cfg.TEXT_JSONL_TRAIN)

  [0] label=6 (frog)  caption: "a photo of a frog"
  [1] label=9 (truck)  caption: "a photo of a truck"
  [2] label=9 (truck)  caption: "a photo of a truck"
CIFAR OK: 10 classes, 50000 train images


/tmp/ipykernel_10371/918584437.py:7: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  batch = pickle.load(open(root / 'data_batch_1', 'rb'), encoding='bytes')


## 5. Run finetuning (`big_vision.trainers.proj.image_text.siglip`)

**Do not skip this cell.** Text-centric SigLIP training: each step pairs one CIFAR image with one synthetic caption; sigmoid contrastive loss; only the **text tower** (+ `t`, `b`) is updated.

With default settings: **3000 steps × batch 16 ≈ 48k pairs** (~1 epoch over CIFAR train). Expect ~10–15 min on RTX 3070 Ti.

If `FRESH_TRAIN = True` in `finetune_config.py`, the workdir is wiped first so you always train from the base checkpoint.

Progress logs appear below. Checkpoints go to `cfg.WORKDIR`. When this cell finishes, run cell 6 for eval (no kernel restart needed unless you hit OOM).

In [5]:
import importlib
import shutil
importlib.reload(cfg)

if cfg.FRESH_TRAIN and cfg.WORKDIR.exists():
    print('FRESH_TRAIN=True — removing old checkpoints at', cfg.WORKDIR)
    shutil.rmtree(cfg.WORKDIR)
cfg.WORKDIR.mkdir(parents=True, exist_ok=True)

config_arg = ','.join([
    'runlocal',
    f'res={cfg.RES}',
    f'variant={cfg.VARIANT}',
    f'batch_size={cfg.BATCH_SIZE}',
    f'total_steps={cfg.TOTAL_STEPS}',
    f'log_steps={cfg.LOG_STEPS}',
    f'ckpt_steps={cfg.CKPT_STEPS}',
    f'lr={cfg.LEARNING_RATE}',
    f'wd={cfg.WEIGHT_DECAY}',
    f'freeze_image={cfg.FREEZE_IMAGE_TOWER}',
    f'seqlen={cfg.SEQLEN}',
])

# Fresh Python process = clean JAX on GPU (notebook keeps TF on CPU only).
cmd = [
    sys.executable, '-m', 'big_vision.trainers.proj.image_text.siglip',
    '--config', f'big_vision/configs/proj/image_text/siglip2_finetune_local.py:{config_arg}',
    '--workdir', str(cfg.WORKDIR),
]
print('Command:\n ', ' '.join(cmd), '\n')

train_env = os.environ.copy()
train_env['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '0')
train_env.pop('BV_JAX_INIT', None)

proc = subprocess.run(cmd, cwd=os.getcwd(), env=train_env)
if proc.returncode != 0:
    raise RuntimeError(
        f'Training failed (exit {proc.returncode}). '
        'Free GPU: close set_up.ipynb, restart kernel, re-run from cell 1.'
    )
last_ptr = cfg.WORKDIR / 'checkpoint.bv-LAST'
if not last_ptr.is_file():
    raise FileNotFoundError(f'Training finished but no checkpoint at {last_ptr}')
print('Training done. Checkpoint:', last_ptr)
print('Next: run cell 6 to compare base vs finetuned on full CIFAR test set.')

FRESH_TRAIN=True — removing old checkpoints at /home/valenbonas/Documents/Investigacion_doctorado/Siglip2_test/workdirs/siglip2_cifar_test
Command:
  /home/valenbonas/miniconda3/envs/siglip2/bin/python -m big_vision.trainers.proj.image_text.siglip --config big_vision/configs/proj/image_text/siglip2_finetune_local.py:runlocal,res=224,variant=B/16,batch_size=16,total_steps=3000,log_steps=25,ckpt_steps=500,lr=0.0001,wd=0.01,freeze_image=True,seqlen=64 --workdir /home/valenbonas/Documents/Investigacion_doctorado/Siglip2_test/workdirs/siglip2_cifar_test 



I0520 20:08:58.128182 139669423637120 xla_bridge.py:895] Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
I0520 20:08:58.129608 139669423637120 xla_bridge.py:895] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0520 20:08:58.140485 139669423637120 siglip.py:94] Hello from process 0 holding 1/1 devices and writing to workdir /home/valenbonas/Documents/Investigacion_doctorado/Siglip2_test/workdirs/siglip2_cifar_test.
I0520 20:08:58.158892 139669423637120 siglip.py:115] NOTE: Initializing train dataset...
I0520 20:08:58.159160 139669423637120 siglip.py:115] NOTE: Global batch size 16 on 1 hosts results in 16 local batch size. With 1 dev per host (1 dev total), that's a 16 per-device batch size.
/home/valenbonas/Documents/Investigacion_doctorado/Siglip2_test/big_vision/big_vision/datasets/cifar10_siglip.py:55: VisibleDeprecationWarning: dtype

Training done. Checkpoint: /home/valenbonas/Documents/Investigacion_doctorado/Siglip2_test/workdirs/siglip2_cifar_test/checkpoint.bv-LAST
Next: run cell 6 to compare base vs finetuned on full CIFAR test set.


## 6. Compare base vs finetuned (CIFAR-10 zero-shot accuracy)

Runs only when `TRAINING_MODE == "legacy_class"`. Uses the same 10 fixed class prompts (`"a photo of a {class}"`) for both models.

By default `finetune_config.py` sets `EVAL_MAX_IMAGES = None` → **all 10,000** CIFAR test images. Set `EVAL_MAX_IMAGES = 2000` for a quicker smoke test.

**Run right after cell 5** — training uses a subprocess that exits and frees the GPU. Restart the kernel only if you get CUDA OOM (~5–10 min for full eval on RTX 3070 Ti).

In [6]:
# Eval — same JAX/model stack as set_up.ipynb (run AFTER cell 5; restart kernel if OOM)
import importlib
importlib.reload(cfg)  # pick up EVAL_MAX_IMAGES / other edits from finetune_config.py

if cfg.TRAINING_MODE != 'legacy_class':
    print('Skipping CIFAR eval (TRAINING_MODE is not legacy_class).')
else:
    import numpy as np
    import jax
    import jax.numpy as jnp
    import ml_collections

    print('jax', jax.__version__, '| devices:', jax.devices(), '| backend:', jax.default_backend())
    import big_vision.utils as u
    import big_vision.models.proj.image_text.two_towers as model_mod
    import big_vision.pp.builder as pp_builder
    import big_vision.pp.ops_general
    import big_vision.pp.ops_image
    import big_vision.pp.ops_text
    import big_vision.pp.proj.paligemma.ops
    RES, SEQLEN = cfg.RES, cfg.SEQLEN
    VARIANT = cfg.VARIANT
    TXTVARIANT = cfg.text_variant()
    EMBDIM = cfg.embed_dim()

    model_cfg = ml_collections.ConfigDict(dict(
        image_model='vit',
        image=dict(pool_type='map', scan=True, variant=VARIANT),
        text_model='proj.image_text.text_transformer',
        text=dict(scan=True, variant=TXTVARIANT, vocab_size=256_000),
        out_dim=[None, EMBDIM],
        bias_init=-10,
    ))
    model = model_mod.Model(**model_cfg)

    pp_img = pp_builder.get_preprocess_fn(f'resize({RES})|value_range(-1, 1)')
    pp_txt = pp_builder.get_preprocess_fn(
        f'lower(key="text")|tok(length={SEQLEN}, model="gemma", bos="no", '
        f'eos="sticky", key="text")'
    )

    import pickle

    root = cfg.CIFAR_ROOT
    meta = pickle.load(open(root / 'batches.meta', 'rb'), encoding='bytes')
    label_names = [x.decode() for x in meta[b'label_names']]

    # Fixed prompts for a fair before/after comparison.
    LABELS = [f'a photo of a {name}' for name in label_names]
    txts = jnp.array([pp_txt({'text': s})['text'] for s in LABELS])

    batch = pickle.load(open(root / 'test_batch', 'rb'), encoding='bytes')
    data = batch[b'data']
    raw_labels = batch[b'labels']

    max_n = cfg.EVAL_MAX_IMAGES
    images, labels = [], []
    for i in range(len(raw_labels)):
        img = data[i].reshape(3, 32, 32).transpose(1, 2, 0)
        images.append(pp_img({'image': img})['image'])
        labels.append(int(raw_labels[i]))
        if max_n and len(images) >= max_n:
            break
    images = np.stack(images)
    labels = np.array(labels)
    n_eval = len(labels)
    print(f'Evaluating on {n_eval} test images ({n_eval / len(raw_labels) * 100:.0f}% of CIFAR test), '
          f'{len(LABELS)} class prompts.')

    def load_base_params():
        return model_mod.load(None, str(cfg.checkpoint_path()), model_cfg)

    def load_finetuned_params():
        ckpt_bv = str(cfg.WORKDIR / 'checkpoint.bv')
        last_ptr = cfg.WORKDIR / 'checkpoint.bv-LAST'
        if not last_ptr.is_file():
            raise FileNotFoundError(
                f'No finetuned checkpoint at {last_ptr}. Run training (cell 5) first.'
            )
        template = load_base_params()
        loaded = u.load_checkpoint_ts(ckpt_bv, tree={'params': template})
        return loaded['params']

    def make_predict(params):
        _, ztxt, out = model.apply({'params': params}, None, txts)
        t, b = out['t'], out['b']

        @jax.jit
        def predict_batch(imgs):
            zimg, _, _ = model.apply({'params': params}, imgs, None)
            return jnp.argmax(zimg @ ztxt.T * t + b, axis=1)

        return predict_batch

    def accuracy(params, batch_size=64, label=''):
        predict_batch = make_predict(params)
        correct, total = 0, 0
        per_class = {i: [0, 0] for i in range(len(LABELS))}  # [correct, total]
        for start in range(0, len(images), batch_size):
            batch_imgs = jnp.array(images[start:start + batch_size])
            preds = predict_batch(batch_imgs)
            y = labels[start:start + batch_size]
            preds_np = np.asarray(preds)
            correct += int((preds_np == y).sum())
            total += len(y)
            for pred, gt in zip(preds_np, y):
                per_class[int(gt)][1] += 1
                if pred == gt:
                    per_class[int(gt)][0] += 1
            done = min(start + batch_size, len(images))
            if label and done % 2048 < batch_size:
                print(f'  [{label}] {done}/{len(images)} — running acc {100 * correct / total:.2f}%')
        return correct / total, per_class

    print('Loading base checkpoint…')
    base_acc, base_pc = accuracy(load_base_params(), label='base')
    print('Loading finetuned checkpoint…')
    ft_acc, ft_pc = accuracy(load_finetuned_params(), label='finetuned')

    print()
    print('=' * 50)
    print(f'Base model accuracy:      {base_acc * 100:.2f}%')
    print(f'Finetuned accuracy:       {ft_acc * 100:.2f}%')
    print(f'Delta:                    {(ft_acc - base_acc) * 100:+.2f} pp')
    print('=' * 50)

    print()
    print('Per-class accuracy (prompt: "a photo of a {class}"):')
    print(f'{"class":<12} {"base":>8} {"finetuned":>10} {"delta":>8}')
    print('-' * 42)
    for i, name in enumerate(label_names):
        b_tot = base_pc[i][1] or 1
        f_tot = ft_pc[i][1] or 1
        b = base_pc[i][0] / b_tot * 100
        f = ft_pc[i][0] / f_tot * 100
        print(f'{name:<12} {b:7.2f}% {f:9.2f}% {f - b:+7.2f} pp')

    print()
    if ft_acc > base_acc:
        print('Finetuning improved zero-shot on this prompt set.')
    elif ft_acc == base_acc:
        print('No overall change — check per-class table; short runs often move only a few classes.')
    else:
        print('Finetuned is lower overall — try more steps or check training loss.')

jax 0.4.33 | devices: [CudaDevice(id=0)] | backend: gpu


/tmp/ipykernel_10371/936339748.py:52: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  batch = pickle.load(open(root / 'test_batch', 'rb'), encoding='bytes')


Evaluating on 10000 test images (100% of CIFAR test), 10 class prompts.
Loading base checkpoint…
  [base] 2048/10000 — running acc 94.73%
  [base] 4096/10000 — running acc 93.75%
  [base] 6144/10000 — running acc 93.65%
  [base] 8192/10000 — running acc 93.40%
Loading finetuned checkpoint…
  [finetuned] 2048/10000 — running acc 95.36%
  [finetuned] 4096/10000 — running acc 94.48%
  [finetuned] 6144/10000 — running acc 94.56%
  [finetuned] 8192/10000 — running acc 94.26%

Base model accuracy:      93.58%
Finetuned accuracy:       94.31%
Delta:                    +0.73 pp

Per-class accuracy (prompt: "a photo of a {class}"):
class            base  finetuned    delta
------------------------------------------
airplane       90.00%     95.50%   +5.50 pp
automobile     97.10%     97.70%   +0.60 pp
bird           92.50%     89.50%   -3.00 pp
cat            91.50%     90.20%   -1.30 pp
deer           91.20%     94.80%   +3.60 pp
dog            91.20%     91.70%   +0.50 pp
frog           89.50